In [13]:
%matplotlib inline

In [14]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt


# Bulgarian Electricity Prices: A Statistical Analysis (2016–2025)

## 1. Introduction

Electricity is the most time-sensitive commodity in a modern economy. Unlike grain or oil, it cannot be stored cheaply at scale; supply must equal demand at every instant, and the price that emerges from this constraint reflects the marginal cost of producing the last megawatt-hour needed to keep the grid balanced. When the weather changes, both sides of that balance shift — heating and cooling demand respond to temperature, wind generation responds to wind, hydro reservoirs respond to precipitation — and the wholesale price moves in response. The relationship between weather and wholesale electricity price is one of the most direct, observable links between the natural environment and a market.

This project examines that relationship in the Bulgarian context, using daily data from October 2016 through December 2025.
### 1.1 The Bulgarian energy market context

Bulgaria's electricity market is small by European standards, but it has all the structural features of a modern liberalised market: a wholesale exchange (IBEX), participation in the European day-ahead coupling mechanism, a generation mix that is geographically and technologically varied (nuclear, lignite, hydro, wind, solar), and exposure to international gas and carbon prices. It is also a market in transition. Over the analysis window, Bulgaria has lived through the launch of its day-ahead exchange, the COVID demand shock, the 2021–2022 European energy crisis, and the post-crisis adjustment.

Two features of the Bulgarian market shape this analysis specifically. The wholesale day-ahead price — the variable studied here — is set on IBEX, which began clearing trades on 1 October 2016. This launch date sets the lower bound of the analytical window: there is no Bulgarian day-ahead price data before October 2016 because no liquid day-ahead market existed before then. Separately, household electricity prices in Bulgaria are largely regulated by KEVR (the Energy and Water Regulatory Commission) rather than set by the market. This means the weather-to-price relationship visible in the data describes how the wholesale market responds to weather, not how household bills respond. Section 2.1 develops this distinction in detail.

### 1.2 Literature Review

The relationship between weather variables and electricity prices is well-established in energy economics research. Boogen et al. (2024) estimate the effect of temperature, wind speed, solar radiation, and precipitation on wholesale electricity prices across six European countries, finding that weather impacts prices in a nonlinear manner and identifying thresholds at which weather effects amplify significantly. Haluška et al. (2025) analyze meteorological variables including temperature, wind speed, and humidity on electricity prices across Central European countries using ENTSO-E data from 2019 to 2024, finding that results are country-specific and asymmetric. Trebbien et al. (2024) provide a comprehensive analysis of patterns and correlations in European day-ahead electricity prices between 2019 and 2023, noting the period was characteristically abnormal due to the energy crisis and that weather is a common driving factor across bidding zones.

No study to our knowledge focuses specifically on Bulgaria over a decade-long period that includes the 2021–2022 energy crisis. This analysis aims to fill that gap.

### 1.3 Research Questions
The analysis is built around three explicitly numbered questions:

1. **How strongly do weather variables correlate with real (inflation-adjusted) Bulgarian day-ahead electricity prices?**
2. **Which weather variable has the strongest independent relationship with price?**
3. **Did the 2021–2022 energy crisis fundamentally alter the weather–price relationship, and if so, how?**

The first question establishes the basic statistical relationship. The second sharpens it by separating out variables that are correlated *because* they correlate with each other (temperature and cloud cover, for example) from variables that contribute independently. The third treats the energy crisis as a natural experiment: a period when external shocks to gas prices and supply disrupted the normal relationship between domestic conditions and domestic prices, and asks whether the disruption persisted, reversed, or simply faded.

### 1.4 Scope and Limitations

This is a correlational study, not a causal one. Weather and price are observed together in the data; nothing here establishes that weather *causes* price changes, only that the two move together (or fail to) under the conditions present in the dataset. The mechanisms by which weather affects electricity prices — heating and cooling demand, renewable generation, fuel substitution — are well understood physically and economically, and the correlations measured here are consistent with those mechanisms. But the analysis itself cannot rule out confounding factors that move with weather (seasonal demand patterns, fuel price cycles, holiday effects), and it does not attempt to.

The wholesale–retail distinction noted above also bounds the analysis. The findings describe how the *Bulgarian wholesale market* prices weather risk, not how Bulgarian households experience weather-driven price changes at the meter. The retail tariff, by design, insulates households from short-term wholesale movements.

Finally, the analysis is observational. The data records what happened; it does not say what would have happened under different conditions. Statements about the energy crisis "altering the relationship" describe what is visible in the before-and-after data, not what would have been true in a counterfactual world without the crisis.

### 1.5 Assumptions

This analysis rests on a small number of explicit assumptions. Each is stated here so that any reader can evaluate whether it holds, and so that any limitation in the conclusions can be traced back to the assumption that produced it.

- **Wholesale market prices reflect supply-and-demand response to weather.** The day-ahead clearing price is treated as the variable on which weather conditions can plausibly act. Regulated retail tariffs and long-term bilateral contracts are not part of this analysis.
- **The five chosen cities are an adequate proxy for national weather.** Sofia, Plovdiv, Varna, Burgas, and Ruse are assumed to span Bulgaria's main population centres and climate zones well enough to construct a national index. Smaller cities and rural areas are not directly represented.
- **Population is an acceptable proxy for electricity demand.** Cities with more people are assumed to contribute more to national electricity demand. The true weighting — by actual electricity consumption — is not publicly available at the regional level.
- **Population weights are stable enough to fix at a single reference year.** Bulgarian city populations did shift over the analysis window, but the shifts are assumed to be small relative to the underlying proxy uncertainty. Static weights anchored to 2020 are used throughout.
- **HICP captures the inflation relevant to euro-denominated electricity prices.** The Bulgarian HICP is assumed to be the appropriate deflator. A producer price index or energy-specific deflator might give different real-price values; the choice of consumer-side HICP is consistent with how end users experience prices.
- **ERA5 reanalysis is treated as ground truth for historical weather.** ERA5 is a model output rather than a direct measurement, but it is assumed to represent past weather faithfully enough for the daily, country-aggregated resolution used here.
- **Daily resolution is fine enough to capture the weather–price relationship.** Intraday dynamics — the morning–evening peak structure, hourly weather extremes — are aggregated away. The research questions are about multi-year patterns, and daily resolution is assumed sufficient for those.
- **The October 2016 – December 2025 window contains enough variation to support the analysis.** The window deliberately includes the COVID shock and the 2021–2022 energy crisis. These are treated as natural experiments rather than excluded as outliers, on the assumption that what the analysis can learn from them is more valuable than the cleanliness lost by including them.
- **Statistical association is the claim being made — not causation.** Every result in this analysis is correlational. Weather and price are observed together; nothing here establishes that weather *causes* price changes, only that the two move together (or fail to) in the data.

## 2. Data Sources

This analysis draws on three independent data sources, combined to form a single daily time series spanning October 2016 through December 2025. The three sources answer different questions that the others cannot: *what did electricity cost on the wholesale market?*, *what was the weather?*, and *what was a euro actually worth?* Eurostat additionally provides population data for the five cities used to construct the national weather index, described alongside the HICP. No single source could answer all of these questions, and substituting any one of them with a derived or modeled equivalent would compromise the analysis.

### 2.1 Source 1 — Ember: Day-Ahead Electricity Prices

Ember is an independent climate and energy think tank that publishes a curated dataset of European wholesale day-ahead electricity prices, sourced originally from ENTSO-E's Transparency Platform. The dataset provides daily clearing prices in nominal euros per megawatt-hour (€/MWh) for Bulgaria.

The data series for Bulgaria begins on 1 October 2016 — the date IBEX, the Bulgarian Independent Energy Exchange, launched and began clearing day-ahead trades. There is no Bulgarian day-ahead price data before this date because no liquid wholesale day-ahead market existed in Bulgaria before then. This sets the lower bound of the analysis window.

The day-ahead market is the central price-discovery mechanism for *wholesale* electricity in Bulgaria. Each day, generators and large buyers submit bids for the following day's hourly delivery, and IBEX clears them into a single price per hour. Ember aggregates these hourly clearing prices into a daily mean, which is the variable used throughout this notebook. Daily resolution is appropriate here because the weather data will also be aggregated to daily resolution, and because the research questions concern multi-year patterns rather than intraday dynamics.

It is worth being explicit about what this price represents and what it does not. The day-ahead price is the wholesale market clearing price — what generators receive for selling electricity to large buyers (utilities, industrial consumers, traders) one day in advance. It reflects the marginal cost of the last unit of electricity needed to meet expected demand, typically a gas-fired plant during peak hours. It does not reflect what Bulgarian households pay on their electricity bills. Household prices in Bulgaria are largely regulated by KEVR (the Energy and Water Regulatory Commission), updated only a few times per year, and include network charges, taxes, and supplier margins on top of the underlying energy cost. Long-term bilateral contracts negotiated outside the exchange are also not reflected in this dataset.

This distinction matters for interpreting the results. The analysis will reveal how the *wholesale market* prices weather risk — how generators and buyers respond to changing supply and demand conditions driven by temperature, wind, precipitation, and so on. It will not reveal how weather affects what consumers pay at the meter, because the regulated retail tariff insulates Bulgarian households from short-term wholesale fluctuations. The wholesale market is the appropriate place to study weather-price relationships precisely because it is the part of the system that responds to weather; the retail tariff, by design, smooths these signals out.

- **Provider:** Ember
- **License:** CC-BY-4.0 (free reuse with attribution)
- **URL:** [ember-energy.org/data/european-wholesale-electricity-price-data](https://ember-energy.org/data/european-wholesale-electricity-price-data)
- **Original source:** ENTSO-E Transparency Platform (Bulgarian market: IBEX)
- **Variable used:** Daily mean day-ahead clearing price, nominal €/MWh
- **Coverage:** 2016-10-01 through 2025-12-31
### 2.2 Source 2 — Open-Meteo: Historical Weather

Open-Meteo provides a free historical weather API that serves reanalysis data from the ERA5 dataset, produced by the European Centre for Medium-Range Weather Forecasts (ECMWF) under the Copernicus Climate Change Service. Reanalysis data is not raw observation — it is the output of a physics-based atmospheric model that ingests observations from weather stations, satellites, and balloons, and produces a globally consistent gridded estimate of past weather. This is the standard approach for historical climate research and is preferable to relying on a single weather station, which may have gaps, instrument changes, or local microclimate effects unrepresentative of the broader region.

Five variables are pulled at hourly resolution and aggregated to daily values: temperature (mean), precipitation (sum), wind speed (mean), cloud cover (mean), and snowfall (sum). The aggregation method is chosen to match the physical meaning of each variable — precipitation and snowfall accumulate, while temperature, wind, and cloud cover are averaged.

Although the Open-Meteo source has weather data going back decades, it is filtered to October 2016 onward to align with the price series. Weather observations from periods when no price data exists cannot enter the analysis, since the analysis is built around the weather–price relationship.

#### Why five cities, not one

Bulgarian electricity demand is not uniform across the country. A national-scale weather variable is needed to correlate with a national-scale price. Using only Sofia would bias the analysis toward conditions in the capital and miss the contributions of coastal cities (where summer cooling demand peaks) and the Danube plain (which has its own climate regime). Weather is therefore pulled for five Bulgarian cities — Sofia, Plovdiv, Varna, Burgas, and Ruse — chosen to span the country's main population centres and climate zones. The cities are then combined into a single national weather index. The construction of that index, including the weighting scheme and its limitations, is documented in Section 3.

- **Provider:** Open-Meteo (Historical Weather API)
- **License:** CC-BY-4.0
- **URL:** [open-meteo.com/en/docs/historical-weather-api](https://open-meteo.com/en/docs/historical-weather-api)
- **Underlying dataset:** ERA5 reanalysis (ECMWF / Copernicus Climate Change Service)
- **Variables used:** Temperature, precipitation, wind speed, cloud cover, snowfall
- **Resolution:** Hourly, aggregated to daily
- **Cities:** Sofia, Plovdiv, Varna, Burgas, Ruse
- **Coverage:** 2016-10-01 through 2025-12-31
### 2.3 Source 3 — Eurostat: Inflation and Population

Eurostat is the statistical office of the European Union. It publishes harmonised statistics across member states, drawing on data collected by each country's National Statistical Institute — for Bulgaria, the National Statistical Institute (NSI Bulgaria). Two pieces of data from Eurostat are used in this analysis. They serve different purposes but share a single publisher, a single license, and a single citation chain, and so are described together here. The HICP is loaded as a CSV; the population values are small enough to be entered inline in the notebook (five numbers, held fixed across the analysis window), with the Eurostat source cited at the point of use.
#### 2.3.1 HICP — Harmonised Index of Consumer Prices

The HICP is the official inflation measure published by Eurostat for all EU member states. National HICP data are collected by NSI Bulgaria and harmonised by Eurostat under a common methodology that makes the indices comparable across countries. The harmonised methodology is the appropriate index basis here, because Bulgarian wholesale electricity is sold on euro-denominated markets, and the HICP reflects the same currency basis as the price data.

The role of this dataset in the analysis is structural, not exploratory. Nominal prices in 2025 cannot be compared directly to nominal prices in 2016, because a euro in 2025 buys substantially less than a euro in 2016. Failing to deflate the price series would mean that any apparent long-term price trend would conflate two distinct phenomena: real changes in the cost of electricity (the thing we want to study) and the general erosion of the euro's purchasing power (which has nothing to do with weather). Section 3 documents the deflation procedure and shows the difference it makes to the price series.

The HICP is published monthly. Daily prices are deflated using the HICP value for the month in which each price observation falls — this is the standard approach and is appropriate given that the index does not change at daily resolution.

- **Variable used:** Monthly HICP index for Bulgaria, base year 2015 = 100
- **URL:** [ec.europa.eu/eurostat/databrowser/view/PRC_HICP_MIDX](https://ec.europa.eu/eurostat/databrowser/view/PRC_HICP_MIDX)
- **Coverage:** 2016-10 through 2025-12
#### 2.3.2 City Population

Population data for the five cities used in the weather index — Sofia, Plovdiv, Varna, Burgas, and Ruse — comes from Eurostat's Urban Audit (dataset `urb_cpop1`). It is used in a supporting role only: to assign weights when combining the five city-level weather series into a single national weather index. Population is used as a proxy for electricity demand, on the reasoning that cities with more people contribute more to national demand. This is a defensible-not-correct choice, and its limitations are discussed in Section 3.

A single reference year (2020) is used to compute the weights, which are then held fixed across the entire analysis period. Bulgarian city populations did shift modestly across the window — Sofia gained residents while smaller cities lost them — but these shifts are small relative to the proxy nature of population-as-demand-weight. Allowing the weights to vary year by year would couple the weather index to demographic drift in a way that complicates the interpretation of the index across time. Holding the weights fixed prioritises a stable, interpretable index over marginal demographic precision. This decision is documented explicitly in Section 3.

The five population values were pulled once from the Eurostat API and saved as a small CSV in the project's `data/` directory, alongside the other source files. Subsequent notebook runs read from this CSV directly. The reproducibility chain is intact — anyone can regenerate the CSV by re-running the original Eurostat fetch (preserved in the notebook's git history), and the values themselves are referenced by both Eurostat dataset code and city code in the source comment.

- **Variable used:** Population for Sofia, Plovdiv, Varna, Burgas, Ruse, reference year 2020
- **Source dataset:** [ec.europa.eu/eurostat/databrowser/view/URB_CPOP1](https://ec.europa.eu/eurostat/databrowser/view/URB_CPOP1)
- **Local cache:** `data/city_population_2020.csv`

### 2.4 Why these sources are independent and appropriate

The project requires at least two independent data sources; this analysis uses three. Each is produced by a different organisation, using different methods, on different phenomena. Ember curates market data from ENTSO-E. Open-Meteo serves ERA5 reanalysis from ECMWF. Eurostat publishes harmonised statistics compiled by national statistical institutes.

Independence matters here because the analysis hinges on relationships *between* sources. If the weather data and the price data came from a common underlying model, any correlation found between them could be an artifact of the shared source rather than a genuine market signal. The two sources whose relationship is the central object of study — Ember (prices) and Open-Meteo (weather) — are produced by completely separate organisations using completely separate methods, with no shared upstream pipeline.

All three sources are licensed for free reuse with attribution, which is documented in the References section. The full citation chain — Ember → ENTSO-E → IBEX, Open-Meteo → ERA5 (ECMWF / Copernicus), Eurostat → NSI Bulgaria — is preserved so that any reader can trace back to the original observations.

## 3. Data Loading & Preprocessing

This section documents how the raw data — Ember prices, Open-Meteo weather for five cities, Eurostat HICP, and Eurostat city population — are loaded, cleaned, and combined into a single analytical dataset. In total, eight CSV files are loaded from disk (one price file, five weather files, one HICP file, one population file) drawn from three providers.
### 3.1 Loading the raw files

The seven CSV files are stored in the project's `data/` directory and loaded with `pandas`. The first step on each file is the same: inspect the shape, the column types, the date range, and the count of missing values per column. This is not a formality — each dataset has its own quirks, and the inspection step is what surfaces them before they cause problems downstream.

The Ember price file is a *multi-country* dataset — it contains day-ahead clearing prices for every European country in a single table, with a country column distinguishing them. Since this analysis concerns Bulgaria only, the file is filtered to Bulgarian rows immediately on load, before any inspection or transformation. Filtering at the loading stage rather than later means every subsequent step — missingness checks, deflation, merging — operates on the variable actually being studied, and the count of NaNs in Section 3.2 reflects gaps in the Bulgarian series specifically rather than gaps that could belong to any of the 30+ countries in the raw file.

After filtering, the Ember data contains daily day-ahead clearing prices for Bulgaria in nominal €/MWh, indexed by date. The five Open-Meteo files (one per city) contain daily weather observations — see Section 3.3 for how the hourly source data was aggregated. The Eurostat HICP file contains the monthly Bulgarian inflation index, with the value held constant across each month. The Eurostat population file contains 2020 population for the five cities used in the weather index — these values are loaded here alongside the other sources, and the weights they produce are constructed in Section 3.4.

The analytical window runs from 1 October 2016 through 31 December 2025. The starting boundary is determined by the Bulgarian day-ahead price series itself: IBEX, the Bulgarian Independent Energy Exchange, launched on 1 October 2016, and there is no Bulgarian wholesale day-ahead price data before that date. Weather, HICP, and population data are all available from earlier years, but they cannot enter the analysis without a corresponding price observation, so the merged dataset is bounded by the price series. No data is discarded for convenience — the start date reflects the actual existence of the market being studied. The choice to begin in October 2016 (a partial year) rather than rounding up to January 2017 is deliberate: the three months at the start represent real observations of a real market, and excluding them would trade transparency for cosmetic round numbers.

Date columns are parsed into proper `datetime` types at load time rather than left as strings. This is the single most important preprocessing step for a time-series project: every subsequent merge, filter, and groupby depends on dates being comparable as dates, not as strings that happen to look like dates.

In [40]:
electricity_price = pd.read_csv("data/european_wholesale_electricity_price_data_daily.csv")
electricity_price['Date'] = pd.to_datetime(electricity_price['Date'])
electricity_price = electricity_price[electricity_price["Country"] == "Bulgaria"]
electricity_price = electricity_price[electricity_price['Date'] < '2026-01-01']
print(f"Prices: {electricity_price.shape}, {electricity_price['Date'].min()} to {electricity_price['Date'].max()}")

Prices: (3379, 4), 2016-10-01 00:00:00 to 2025-12-31 00:00:00


In [43]:
cities = ['sofia', 'plovdiv', 'varna', 'burgas', 'ruse']
weather = {c: pd.read_csv(f'data/weather_{c}.csv', parse_dates=['date']) for c in cities}
for c in cities:
    weather[c] = weather[c][weather[c]['date'].between('2015-01-01', '2025-12-31')]
    print(f"Weather ({c}): {weather[f'{c}'].shape}")

Weather (sofia): (4018, 6)
Weather (plovdiv): (4018, 6)
Weather (varna): (4018, 6)
Weather (burgas): (4018, 6)
Weather (ruse): (4018, 6)


In [49]:
hicp = pd.read_csv('data/prc_hicp_midx_page_linear.csv', parse_dates=['TIME_PERIOD'])
hicp = hicp[hicp['TIME_PERIOD'].between('2015-01-01', '2025-12-31')]
print(f"HICP: {hicp.shape}")

HICP: (132, 10)


In [88]:
populations = pd.read_csv("data/city_population_2020.csv")
populations

,city,population
0,Sofia,1242568
1,Plovdiv,347851
2,Varna,336216
3,Burgas,201779
4,Ruse,141231


### 3.2 Per-source missingness check

Before any transformation, each loaded dataset is checked for missing values on its own terms. This is the first of the two missingness moments described above: missingness that exists in the raw source itself, independent of any downstream processing. The point of doing this *before* any further work is that subsequent steps — aggregation, weighting, deflation, merging — will obscure the origin of any NaN they encounter, and it becomes impossible to diagnose missingness once it has been mixed into a derived column.

Each source is checked separately, and the response to any missingness depends on what the missingness *means* in that source.

- **Ember prices.** Occasional days are missing from the Bulgarian day-ahead price series. These correspond to gaps in ENTSO-E reporting rather than to days on which the market failed to clear. Imputation is rejected for this source: substituting a fictitious price into the variable being studied would compromise the analysis. Missing price days are flagged here and dropped from the merged dataset later in Section 3.6.
- **Open-Meteo weather.** Reanalysis data is gridded model output with global coverage; per-source missingness is not expected, and inspection confirms none is present in any of the five city files. No action is required.
- **Eurostat HICP.** The Bulgarian HICP series is complete across the analysis window. No action is required.
- **Eurostat population.** Annual population values for the five cities are present. No action is required.

The result of this step is a clear inventory: *if a NaN appears later in the pipeline, the price series is the only place it could have originated, and the count of price-side gaps is known up front*. This will matter when interpreting the post-merge check in Section 3.6.

In [89]:
electricity_price.isnull().sum()

Country             0
ISO3 Code           0
Date                0
Price (EUR/MWhe)    0
dtype: int64

In [90]:
for c in cities: 
    print(weather[c].isnull().sum()) 

date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64
date                0
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
cloudcover_pct      0
snowfall_cm         0
dtype: int64


Drop empty Eurostat metadata columns (OBS_FLAG, CONF_STATUS).
These are SDMX export defaults; no observations in this window are flagged.

In [91]:
hicp.drop("OBS_FLAG" , axis='columns', inplace=True)
hicp.drop("CONF_STATUS", axis='columns', inplace=True)

KeyError: "['OBS_FLAG'] not found in axis"

In [92]:
hicp.isnull().sum()

DATAFLOW       0
LAST UPDATE    0
freq           0
unit           0
coicop         0
geo            0
TIME_PERIOD    0
OBS_VALUE      0
dtype: int64

### 3.3 Daily weather data

The Open-Meteo source provides weather observations at hourly resolution. For this project, the hourly data was aggregated to daily resolution outside the notebook before loading, and the daily CSVs are what enters the analysis here. This was a practical choice: the hourly files are large, the analysis operates entirely at daily resolution, and re-aggregating on every notebook run would add substantial loading time without changing any result.

The aggregation method was chosen variable by variable to match the physical meaning of each one:

- **Temperature** is averaged across the 24 hourly values for the day. The daily mean is the most common choice in climate and energy studies.
- **Precipitation** is summed. A daily precipitation total of 10 mm is meaningful; a daily precipitation average is not.
- **Snowfall** is summed, for the same reason as precipitation.
- **Wind speed** is averaged. Daily peak gust would be physically meaningful too, but mean wind speed is the standard variable for energy-system studies because wind generation depends on sustained wind, not on isolated gusts.
- **Cloud cover** is averaged. Cloud cover is reported as a percentage at each hour, and the daily mean represents the share of the day that was cloudy.

This step was performed independently for each of the five cities, producing five daily CSVs that are loaded and merged in the notebook itself.

### 3.4 Constructing the national weather index

The five city-level daily weather series are now combined into a single national weather series. The combination uses a weighted mean: each city's weather contributes to the national value in proportion to its population.

Population data was loaded in Section 3.1 alongside the other source files. Here, those values are converted into weights — each city's weight is its population divided by the total population across the five cities — and the weights are then applied to the daily weather series. The formula: for any weather variable on any day, the national value is the sum across the five cities of `weight × city_value`, where the weights sum to one.

In [112]:
populations['weight'] = populations['population'] / populations['population'].sum()
assert populations.weight.sum() - 1 == 0
populations.head()

,city,population,weight
0,Sofia,1242568,0.547472
1,Plovdiv,347851,0.153262
2,Varna,336216,0.148136
3,Burgas,201779,0.088903
4,Ruse,141231,0.062226


### 3.5 Deflating prices to real terms

The Ember price series is in nominal €/MWh — the actual euro figure that changed hands on the day. To compare prices across the window meaningfully, these nominal values must be converted to real terms, removing the effect of general inflation.

The deflation formula used is the standard one:

> Real Price = Nominal Price × (HICP_base / HICP_t)

Where HICP_t is the HICP value for the month in which the price observation falls, and HICP_base is the HICP value for the chosen base year. The base year used here is 2015, the standard Eurostat reference period, with the Bulgarian HICP rebased so that the 2015 annual mean equals 100. This is independent of the analysis window: the deflation base year just sets the scale in which real prices are reported, and choosing 2015 keeps the values consistent with how Eurostat publishes the underlying series. The result is a price series expressed in "constant 2015 euros" — every value across the window is denominated in the purchasing power of a 2015 euro.

Two implementation details matter. First, the HICP is published monthly and the prices are daily, so each daily price is deflated using the HICP value for its calendar month. The HICP does not change within a month, so this is the appropriate level of resolution. Second, the deflation is applied as a multiplication, not a subtraction — inflation compounds, and adjusting for it correctly requires multiplicative scaling.

A small note on what the choice of HICP commits us to: HICP is the *consumer* price index, capturing inflation as households experience it through the prices they pay for a basket of goods. There are alternative deflators — the producer price index, an energy-specific deflator, or the GDP deflator — that would yield slightly different real-price values. The choice of HICP here is consistent with the framing of the project: we are interested in how the wholesale electricity price moved over time relative to the general purchasing power of the euro, not relative to the cost basket of producers or to the broader economy. This is a defensible-not-correct choice, and a reader who wanted to interrogate the analysis with a different deflator could swap it in by replacing one CSV.

The effect of deflation on the price series is shown in Section 4, side by side with the nominal series. The visual comparison makes the case for deflation more directly than any verbal argument: the nominal series suggests that prices in 2025 were higher than at the start of the window partly because of real cost increases and partly because the euro itself is worth less; the real series isolates the first effect from the second.